# Phase 1 Capstone: Mobile App Crash Log Analyzer

**Objective:** Use Python, NumPy, Pandas, and Matplotlib to analyze synthetic mobile crash logs.

**Skills practiced:** Data loading, cleaning, EDA, visualization, statistics


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random

sns.set_theme(style='darkgrid')
print('Libraries loaded successfully')

## Step 1: Generate Synthetic Crash Log Data

In a real scenario this would come from Firebase Crashlytics, Sentry, or a custom logging system.


In [ ]:
np.random.seed(42)
random.seed(42)

n_events = 5000

os_versions = ['iOS 16.0', 'iOS 16.1', 'iOS 17.0', 'Android 12', 'Android 13', 'Android 14']
app_versions = ['2.1.0', '2.2.0', '2.3.0', '2.3.1', '2.4.0']
device_types = ['iPhone 13', 'iPhone 14', 'iPhone 15', 'Pixel 7', 'Galaxy S23', 'OnePlus 11']
crash_types = ['NullPointerException', 'OutOfMemoryError', 'NetworkTimeout', 'UIThreadViolation', 'DatabaseLocked']

# Simulate that version 2.3.0 had a bad bug (higher crash rate)
version_crash_weight = {'2.1.0': 0.05, '2.2.0': 0.04, '2.3.0': 0.20, '2.3.1': 0.06, '2.4.0': 0.03}

base_date = datetime(2025, 1, 1)

records = []
for i in range(n_events):
    app_v = random.choice(app_versions)
    is_crash = np.random.random() < version_crash_weight[app_v]
    records.append({
        'event_id': f'EVT_{i:05d}',
        'timestamp': base_date + timedelta(days=np.random.randint(0, 90), hours=np.random.randint(0, 24)),
        'app_version': app_v,
        'os_version': random.choice(os_versions),
        'device': random.choice(device_types),
        'is_crash': is_crash,
        'crash_type': random.choice(crash_types) if is_crash else None,
        'session_duration_s': np.random.exponential(300),
        'memory_mb': np.random.normal(250, 80)
    })

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
df.head()

## Step 2: Data Quality Check


In [ ]:
print('=== Data Info ===')
print(df.info())
print('\n=== Null Counts ===')
print(df.isnull().sum())
print('\n=== Numeric Summary ===')
print(df.describe())

## Step 3: Crash Rate by App Version


In [ ]:
crash_by_version = df.groupby('app_version').agg(
    total_events=('event_id', 'count'),
    total_crashes=('is_crash', 'sum')
).reset_index()
crash_by_version['crash_rate'] = crash_by_version['total_crashes'] / crash_by_version['total_events']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(crash_by_version['app_version'], crash_by_version['crash_rate'] * 100,
              color=['red' if r > 0.1 else 'steelblue' for r in crash_by_version['crash_rate']])
ax.set_xlabel('App Version')
ax.set_ylabel('Crash Rate (%)')
ax.set_title('Crash Rate by App Version')
for bar, rate in zip(bars, crash_by_version['crash_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{rate*100:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print(crash_by_version)

## Step 4: Crash Type Distribution


In [ ]:
crashes_only = df[df['is_crash'] == True]
crash_type_counts = crashes_only['crash_type'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

crash_type_counts.plot(kind='bar', ax=ax1, color='coral')
ax1.set_title('Crash Count by Type')
ax1.set_xlabel('Crash Type')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=30)

ax2.pie(crash_type_counts, labels=crash_type_counts.index, autopct='%1.1f%%', startangle=140)
ax2.set_title('Crash Type Distribution')

plt.tight_layout()
plt.show()

## Step 5: Statistical Analysis


In [ ]:
print('=== Key Statistics ===')
print(f'Total events analyzed: {len(df):,}')
print(f'Total crashes: {df["is_crash"].sum():,}')
print(f'Overall crash rate: {df["is_crash"].mean()*100:.2f}%')
print()

non_crash = df[df['is_crash'] == False]
print(f'Avg session duration (non-crash): {non_crash["session_duration_s"].mean():.1f}s')
print(f'Median session duration: {non_crash["session_duration_s"].median():.1f}s')
print(f'P95 session duration: {np.percentile(non_crash["session_duration_s"], 95):.1f}s')
print()

print(f'Avg memory at crash: {crashes_only["memory_mb"].mean():.1f} MB')
print(f'Avg memory (no crash): {non_crash["memory_mb"].mean():.1f} MB')

# T-test: is memory usage significantly different at crash time?
from scipy import stats
t_stat, p_val = stats.ttest_ind(crashes_only['memory_mb'].dropna(), non_crash['memory_mb'].dropna())
print(f'\nT-test memory crash vs non-crash: t={t_stat:.3f}, p={p_val:.4f}')
if p_val < 0.05:
    print('Memory usage IS significantly different at crash time (p < 0.05)')
else:
    print('No significant memory difference (simulated data — this is expected)')

## Step 6: Time Series — Crash Rate Over Time


In [ ]:
df['date'] = df['timestamp'].dt.date
daily = df.groupby('date').agg(events=('event_id','count'), crashes=('is_crash','sum')).reset_index()
daily['crash_rate'] = daily['crashes'] / daily['events']
daily['date'] = pd.to_datetime(daily['date'])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(daily['date'], daily['crash_rate'] * 100, linewidth=1.5, color='tomato')
ax.fill_between(daily['date'], daily['crash_rate'] * 100, alpha=0.2, color='tomato')
ax.set_title('Daily Crash Rate Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Crash Rate (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Reflection

**What you just did:**
- Used NumPy for random data generation and statistical operations
- Used Pandas for groupby aggregation, filtering, derived column creation
- Used Matplotlib/Seaborn for multi-panel visualizations
- Used SciPy for a t-test (hypothesis testing)
- Applied your mobile domain knowledge to make the data realistic

**What this connects to in AI:**
- EDA is step 1 of every ML project
- Statistical tests are used to validate if model improvements are real
- Crash rate over time is a time-series problem (forecasting)
- Crash type classification could be automated with NLP on stack traces

**Next step:** Move to Phase 2 and learn to build models that predict which sessions will crash.
